In [8]:
TEXT_INPUT="hello world"

In [9]:
def step1_generate_input():
    """Python → bits.txt"""
    print("\n[1/5] Generating input bits...")
    binary = ''.join(format(ord(c), '08b') for c in TEXT_INPUT)
    with open("bits.txt", "w") as f:
        f.write('\n'.join(binary) + '\n')
    print(f"      Text: '{TEXT_INPUT}'")
    print(f"      Bits: {len(binary)} bits written to bits.txt")
    return binary

In [10]:
step1_generate_input()


[1/5] Generating input bits...
      Text: 'hello world'
      Bits: 88 bits written to bits.txt


'0110100001100101011011000110110001101111001000000111011101101111011100100110110001100100'

In [11]:
#to generate pwl file
def generate_pwl(compressed_bits):
    """compressed.txt → signal.pwl"""
    print("\n[3/5] Generating PWL file for LTSpice...")
    lines = []
    t = 0
    for bit in compressed_bits:
        v = V_HIGH if bit == '1' else 0.0
        lines.append(f"{t:.6e} {v:.1f}")
        t += 50e-6
    lines.append(f"{t:.6e} 0.0")
    with open("signal.pwl", "w") as f:
        f.write('\n'.join(lines))
    print(f"      PWL file written: {len(lines)} time points")
    print(f"      Total duration: {t*1e6:.1f} µs")

In [16]:
!pip install ltspice
import numpy as np
import ltspice

  Preparing metadata (setup.py) ... done
  Created wheel for ltspice: filename=ltspice-1.0.6-py3-none-any.whl size=6235 sha256=e811186f8667ae7ddd1ef5c62d4a3648f165a3cb407e6ae644dfa19b585d9bf0
  Stored in directory: /root/.cache/pip/wheels/27/01/bc/8bc0912042857dcc5654b6ed12437bfa9c4865e1b95365a9e2
Successfully built ltspice


In [31]:
import numpy as np
import ltspice

# 1. Helper function to decompress your flat file format correctly
def parse_flat_compressed_file(file_path):
    with open(file_path, "r") as f:
        # Strip white spaces and remove empty lines
        raw_bits = [line.strip() for line in f if line.strip()]

    reconstructed_bits = []
    idx = 0

    # Process the file in sequential blocks of 5 lines (1 bit value + 4 bits for binary count)
    while idx < len(raw_bits):
        if idx + 4 >= len(raw_bits):
            break

        run_bit = raw_bits[idx]
        count_bits = raw_bits[idx+1 : idx+5]

        # Turn the 4 lines of count bits into a string, then parse as base-2 binary integer
        count_str = "".join(count_bits)
        run_count = int(count_str, 2)

        # Re-inflate the stream by repeating the bit by its run count
        reconstructed_bits.extend([run_bit] * run_count)
        idx += 5

    return "".join(reconstructed_bits)

# 2. Extract the true target uncompressed bitstream from your text file
try:
    original_bits = parse_flat_compressed_file("compressed.txt")
    print(f"Successfully reconstructed {len(original_bits)} uncompressed bits from compressed.txt")
except FileNotFoundError:
    print("ERROR: Please make sure 'compressed.txt' is in the same folder as this script.")
    original_bits = ""

# 3. Process the true bitstream back into ASCII text
if original_bits:
    chars = []
    for i in range(0, len(original_bits), 8):
        byte = original_bits[i:i+8]
        if len(byte) == 8:
            val = int(byte, 2)
            # Only map printable standard ASCII characters to filter out padding/junk trailing bits
            if 32 <= val <= 126:
                chars.append(chr(val))

    print("\nRecovered text:", ''.join(chars))
else:
    print("Could not recover text because bitstream reconstruction failed.")

Successfully reconstructed 86 uncompressed bits from compressed.txt

Recovered text: HELLO WORL


In [ ]:
import numpy as np

def parse_ltspice_wave_to_text(spice_file, bit_period=1.0e-6, threshold=0.4):
    print("\n[4/5] Reading analog waveform from LTSpice...")
    
    # Load time and voltage arrays, skipping the text header row
    try:
        time_axis, v_ch = np.loadtxt(spice_file, skiprows=1, unpack=True)
        print(f"      Successfully loaded {len(time_axis)} simulation data points.")
    except FileNotFoundError:
        print(f"ERROR: '{spice_file}' not found! Export it from LTSpice first.")
        return ""

    # Calculate total simulated duration and reconstruct bit count
    total_duration = time_axis[-1]
    num_bits = int(total_duration / bit_period)
    
    demodulated_bits = []
    
    print(f"      Sampling {num_bits} bits at midpoint intervals...")
    for i in range(num_bits):
        
        sample_time = (i * bit_period) + (bit_period / 2.0)
        
        
        closest_idx = np.abs(time_axis - sample_time).argmin()
        sampled_voltage = v_ch[closest_idx]
        
        # Comparator Decision Logic: Absolute value due to negative carrier swing
        if np.abs(sampled_voltage) >= threshold:
            demodulated_bits.append("1")
        else:
            demodulated_bits.append("0")
            
    return "".join(demodulated_bits)


demod_bitstream = parse_ltspice_wave_to_text("demod_output.txt", bit_period=1.0e-6)


if demod_bitstream:
    print("\n[5/5] Translating recovered bitstream back to string...")
    recovered_chars = []
    for i in range(0, len(demod_bitstream), 8):
        byte_chunk = demod_bitstream[i:i+8]
        if len(byte_chunk) == 8:
            ascii_val = int(byte_chunk, 2)
           
            if 32 <= ascii_val <= 126:
                recovered_chars.append(chr(ascii_val))
                
    final_text = "".join(recovered_chars)
    print(f"\n======== END-TO-END VERIFICATION ========")
    print(f"Expected Input: 'HELLO WORLD'")
    print(f"Recovered Text: '{final_text}'")
    print(f"Pipeline Integrity Verified: {final_text == 'HELLO WORLD'}")
    print(f"=========================================")